# Notebook 3 — Defining the NORR → MITT Corridor

**Course: Cross-Corridor Capacity Analysis with pandapower (Svedala grid)**

## What is a corridor?

A **corridor** is the set of branches that connect two zones. The aggregate
active-power flow on those branches — measured in a consistent direction — is
the **corridor flow**. In Sweden the corridor between hydro-rich `ZON_NORR`
and load-heavy `ZON_MITT` is the famous **Snitt 2 / SE2-SE3 boundary**, often
the binding constraint for day-ahead capacity allocation.

## What does "increasing transfer" mean?

We don't change line ratings. We change the **dispatch**: we ramp generators
in `ZON_NORR` *up* and generators in `ZON_MITT` *down* by the same total
amount. The participation factors that distribute the change among generators
are **Generation Shift Keys (GSK)**.

## Learning objectives

- Use the existing `bus.zone` column to define source and sink zones.
- Identify the **corridor branches** that cross the NORR/MITT boundary.
- Compute the **base corridor flow** in the source → sink direction.
- Define **GSK vectors** for the source and sink zones.
- Confirm both zones have enough generation headroom to support a transfer increase.

## 3.1  Set up

In [1]:
import pandapower as pp
import pandas as pd
import numpy as np
import json

net = pp.from_json('data/svedala_base.json')
pp.runpp(net)
print('Base case OK.')

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


Base case OK.


## 3.2  Define source and sink zones

The Svedala model has its zones already populated by the CIM importer — we just pick which two we want.

In [3]:
ZONE_SOURCE = 'ZON_NORR'   # exporter (hydro-rich)
ZONE_SINK   = 'ZON_MITT'   # importer (load-heavy)

# The bidding-zone name is in SubGeographicalRegion_name (CIM column).
# net.bus.zone holds the substation name, not the zone.
ZONE_A_BUSES = net.bus[net.bus.SubGeographicalRegion_name == ZONE_SOURCE].index.tolist()
ZONE_B_BUSES = net.bus[net.bus.SubGeographicalRegion_name == ZONE_SINK].index.tolist()

print(f'Source zone {ZONE_SOURCE}: {len(ZONE_A_BUSES)} buses')
print(f'Sink   zone {ZONE_SINK}:   {len(ZONE_B_BUSES)} buses')


Source zone ZON_NORR: 23 buses
Sink   zone ZON_MITT:   19 buses


## 3.3  Identify the corridor branches

A line/transformer crosses the boundary when its two endpoints are in
different zones.

In [4]:
def cross_zone_branches(net, zone_a, zone_b):
    a, b = set(zone_a), set(zone_b)
    line_dir = []
    for idx, row in net.line.iterrows():
        if row.from_bus in a and row.to_bus in b:
            line_dir.append((idx, +1))
        elif row.from_bus in b and row.to_bus in a:
            line_dir.append((idx, -1))
    trafo_dir = []
    for idx, row in net.trafo.iterrows():
        if row.hv_bus in a and row.lv_bus in b:
            trafo_dir.append((idx, +1))
        elif row.hv_bus in b and row.lv_bus in a:
            trafo_dir.append((idx, -1))
    return line_dir, trafo_dir

line_dir, trafo_dir = cross_zone_branches(net, ZONE_A_BUSES, ZONE_B_BUSES)
print(f'Corridor: {len(line_dir)} line(s), {len(trafo_dir)} trafo(s):')
for idx, d in line_dir:
    fb, tb = net.line.loc[idx, ['from_bus', 'to_bus']]
    print(f'  line  {idx:>3}  {net.line.at[idx,"name"]}  '
          f'(bus {fb:>4}: {net.bus.at[fb,"name"]} → '
          f'bus {tb:>4}: {net.bus.at[tb,"name"]})  sign {d:+d}')
for idx, d in trafo_dir:
    fb, tb = net.trafo.loc[idx, ['hv_bus', 'lv_bus']]
    print(f'  trafo {idx:>3}  {net.trafo.at[idx,"name"]}  '
          f'(bus {fb:>4}: {net.bus.at[fb,"name"]} → '
          f'bus {tb:>4}: {net.bus.at[tb,"name"]})  sign {d:+d}')

Corridor: 5 line(s), 0 trafo(s):
  line    3  CL17  (bus 1014: TORNÅ CT32_400.0kV → bus  346: KÄRNAN FT44_400.0kV)  sign +1
  line   38  CL14  (bus  729: STENFORSEN CT31_400.0kV → bus  130: DALBO FT41_400.0kV)  sign +1
  line   56  CL16  (bus 1014: TORNÅ CT32_400.0kV → bus  544: NORRÅS FT42_400.0kV)  sign +1
  line   62  CL15  (bus  729: STENFORSEN CT31_400.0kV → bus  130: DALBO FT41_400.0kV)  sign +1
  line   67  CL12  (bus  464: NJAGGO CT21_400.0kV → bus  346: KÄRNAN FT44_400.0kV)  sign +1


## 3.4  Base corridor flow

The corridor flow is the sum of active-power flows on corridor branches, all
signed to point in the same direction (we choose source → sink as positive).

In [5]:
def corridor_flow(net, line_dir, trafo_dir):
    p = 0.0
    for idx, d in line_dir:
        p += d * net.res_line.at[idx, 'p_from_mw']
    for idx, d in trafo_dir:
        p += d * net.res_trafo.at[idx, 'p_hv_mw']
    return p

P0 = corridor_flow(net, line_dir, trafo_dir)
direction = 'NORR → MITT' if P0 > 0 else 'MITT → NORR'
print(f'Base corridor flow {ZONE_SOURCE} → {ZONE_SINK}: {P0:+.2f} MW '
      f'({direction})')

Base corridor flow ZON_NORR → ZON_MITT: +2497.07 MW (NORR → MITT)


## 3.5  Generation Shift Keys (GSK)

Pmax-proportional GSK: each generator's participation in the shift is
proportional to its installed capacity within its zone. Most TSOs use
Pmax-proportional or headroom-proportional GSK in practice.

In [6]:
def gsk_pmax(net, zone_buses):
    in_zone = net.gen[net.gen.bus.isin(zone_buses) & net.gen.in_service]
    if in_zone.empty:
        raise ValueError('No in-service generators in this zone.')
    pmax = in_zone.max_p_mw.replace(0, np.nan).fillna(in_zone.p_mw.abs())
    return pmax / pmax.sum()

GSK_A = gsk_pmax(net, ZONE_A_BUSES)
GSK_B = gsk_pmax(net, ZONE_B_BUSES)

# Show the shift keys
gsk_view = pd.DataFrame({
    'gen_name': net.gen.loc[GSK_A.index, 'name'].values,
    'GSK':      GSK_A.round(3).values,
    'zone':     ZONE_SOURCE,
})
gsk_view2 = pd.DataFrame({
    'gen_name': net.gen.loc[GSK_B.index, 'name'].values,
    'GSK':      GSK_B.round(3).values,
    'zone':     ZONE_SINK,
})
pd.concat([gsk_view, gsk_view2], ignore_index=True)

,gen_name,GSK,zone
0,AGGAN_G1,0.064,ZON_NORR
1,AGGAN_G2,0.096,ZON_NORR
2,NJAGGO_G1,0.048,ZON_NORR
3,NORRSELE_G1,0.080,ZON_NORR
4,NORRSELE_G2,0.053,ZON_NORR
5,OLMÅFALLET_G1,0.112,ZON_NORR
6,STENFORSEN_G1,0.056,ZON_NORR
7,STORFORS_G1,0.096,ZON_NORR
8,STORTRÄSK_G1,0.040,ZON_NORR
9,STUPET_G1,0.080,ZON_NORR


## 3.6  Headroom check

There is no point asking for 1000 MW of additional transfer if the source
zone only has 600 MW of free generation capacity.

In [7]:
def zone_headroom_mw(net, zone_buses):
    g = net.gen[net.gen.bus.isin(zone_buses) & net.gen.in_service]
    return float((g.max_p_mw - g.p_mw).clip(lower=0).sum())

def zone_decroom_mw(net, zone_buses):
    g = net.gen[net.gen.bus.isin(zone_buses) & net.gen.in_service]
    minp = g.min_p_mw.fillna(0)
    return float((g.p_mw - minp).clip(lower=0).sum())

head_A = zone_headroom_mw(net, ZONE_A_BUSES)
decr_B = zone_decroom_mw(net, ZONE_B_BUSES)

print(f'{ZONE_SOURCE} free upward capacity: {head_A:7.2f} MW')
print(f'{ZONE_SINK} free downward capacity: {decr_B:7.2f} MW')
print(f'-> Maximum *dispatchable* shift {ZONE_SOURCE} → {ZONE_SINK}: '
      f'{min(head_A, decr_B):7.2f} MW')
print('   (Capacity at the corridor itself is usually the smaller binding limit.)')

ZON_NORR free upward capacity: 1740.68 MW
ZON_MITT free downward capacity: 3420.02 MW
-> Maximum *dispatchable* shift ZON_NORR → ZON_MITT: 1740.68 MW
   (Capacity at the corridor itself is usually the smaller binding limit.)


## 3.7  Save the corridor definition

In [8]:
corridor = {
    'name':              f'{ZONE_SOURCE}_to_{ZONE_SINK}',
    'zone_source':       ZONE_SOURCE,
    'zone_sink':         ZONE_SINK,
    'zone_a_buses':      ZONE_A_BUSES,
    'zone_b_buses':      ZONE_B_BUSES,
    'corridor_lines':    [(int(i), int(d)) for i, d in line_dir],
    'corridor_trafos':   [(int(i), int(d)) for i, d in trafo_dir],
    'gsk_a':             {int(k): float(v) for k, v in GSK_A.items()},
    'gsk_b':             {int(k): float(v) for k, v in GSK_B.items()},
    'base_flow_a_to_b':  float(P0),
}
with open('data/corridor.json', 'w') as f:
    json.dump(corridor, f, indent=2, ensure_ascii=False)
print('Saved data/corridor.json')

Saved data/corridor.json


## 3.8  Exercises

1. Repeat the analysis for the **MITT → SYDVÄST** corridor (the SE3-SE4
   boundary). How many corridor branches does it have? What is the base
   flow direction?
2. Compare **Pmax-proportional GSK** with **uniform GSK** (each in-service
   generator gets the same share). For ZON_NORR specifically, which
   generators dominate under Pmax GSK? Are those the *cheapest* hydro units
   the market would actually dispatch first?

In [ ]:
# Exercise 1 — MITT → SYDVÄST corridor (SE3-SE4 boundary)
MITT_buses   = net.bus[net.bus.SubGeographicalRegion_name == 'ZON_MITT'].index.tolist()
SYDVAST_buses = net.bus[net.bus.SubGeographicalRegion_name == 'ZON_SYDVÄST'].index.tolist()

ld2, td2 = cross_zone_branches(net, MITT_buses, SYDVAST_buses)
P2 = corridor_flow(net, ld2, td2)

print(f'ZON_MITT:    {len(MITT_buses)} buses')
print(f'ZON_SYDVÄST: {len(SYDVAST_buses)} buses')
print(f'Corridor MITT → SYDVÄST: {len(ld2)} line(s), {len(td2)} trafo(s)')
direction2 = 'MITT → SYDVÄST' if P2 > 0 else 'SYDVÄST → MITT'
print(f'Base flow: {P2:+.2f} MW  ({direction2})')


In [ ]:
# Exercise 2 — Compare Pmax-proportional GSK with uniform GSK for ZON_NORR
in_zone_A = net.gen[net.gen.bus.isin(ZONE_A_BUSES) & net.gen.in_service]
pmax_A    = in_zone_A.max_p_mw.replace(0, np.nan).fillna(in_zone_A.p_mw.abs())
gsk_uniform_A = pd.Series(1 / len(in_zone_A), index=in_zone_A.index)

cmp = pd.DataFrame({
    'gen_name':    net.gen.loc[in_zone_A.index, 'name'].values,
    'max_p_mw':    net.gen.loc[in_zone_A.index, 'max_p_mw'].values,
    'GSK_pmax':    (pmax_A / pmax_A.sum()).round(3).values,
    'GSK_uniform': gsk_uniform_A.round(3).values,
}).sort_values('GSK_pmax', ascending=False)

print(cmp.to_string(index=False))
print()
print('Under Pmax GSK the large-capacity units (OLMÅFALLET, AGGAN, STORFORS, etc.) dominate.')
print('Uniform GSK spreads the redispatch equally across all in-service generators.')


---

✅ **Checkpoint reached.** Corridor defined, GSK ready.

Continue to [Notebook 4 — One Transfer Step](04_single_transfer_step.ipynb).